![banner image](./seadam.png)

# Zanddam / dam schuppen candidate dates

We look for low tide later in the afternoon. The idea is: drive in the morning, arrive around 9u30, start around 10u30-11u00, and get a long useful water-flow window.

This notebook keeps the step-by-step visual style, but no longer needs Selenium or Chrome. It calls the same JSON data endpoint that the OD Nature tide website uses behind the table button.

## 1. Setup and parameters

Change `START_DATE`, `END_DATE`, `FUTURE_ONLY`, or `ALLOWED_MONTHS` here when you want another run. The tide times are converted from UTC to `Europe/Brussels`.

In [ ]:
from datetime import date

import pandas as pd
import numpy as np
import requests
from holidays import Belgium

TIDES_URL = "https://odnature.naturalsciences.be/marine-forecasting-centre/nl/ajax/getHarmonicTides"
TIMEZONE = "Europe/Brussels"

# Parameters: change these first when you want another year or another season.
START_DATE = "2026-01-01"
END_DATE = "2026-12-31"
FUTURE_ONLY = True  # True = only show dates from today onward

# Season filter: nice months, but skip the busy summer months.
ALLOWED_MONTHS = [5, 6, 9, 10]


## 2. Download the tide table

The website builds its table from a JSON endpoint. Calling that directly is cleaner and more robust than opening a browser with Selenium.

In [ ]:
def fetch_tides(start_date: str, end_date: str) -> pd.DataFrame:
    """Download harmonic tide predictions as JSON.

    This replaces the older Selenium/Chrome scraping step. It uses the same data
    behind the website button "Tabel actualiseren", but is easier to run in
    Jupyter, on Hermes, and later on another laptop.
    """
    response = requests.get(
        TIDES_URL,
        params={
            "start_date": start_date,
            "end_date": end_date,
            "action": "harmonic-tides",
            "format": "json",
        },
        timeout=30,
    )
    response.raise_for_status()
    return pd.DataFrame(response.json())

raw_df = fetch_tides(START_DATE, END_DATE)
print(f"Downloaded {len(raw_df)} tide records from {START_DATE} until {END_DATE}")
raw_df.head()


## 3. Clean and prepare the data

We rename the columns, convert water levels to numbers, combine date + time, and convert UTC tide times to Belgian local time.

In [ ]:
df = raw_df.copy()
df.columns = ['datum', 'hoog_water', 'hoog_tijd', 'laag_water', 'laag_tijd']

# Replace missing tide values and turn water levels into numbers.
df = df.replace({"--.--": np.nan})
df = df.astype({'hoog_water': 'float', 'laag_water': 'float'})

# The website gives tide times in UTC. Convert them to local Belgian coast time.
df['datum'] = pd.to_datetime(df['datum'])
df['hoog_tijd'] = pd.to_datetime(df['datum'].dt.strftime('%Y-%m-%d') + ' ' + df['hoog_tijd'], errors='coerce')
df['laag_tijd'] = pd.to_datetime(df['datum'].dt.strftime('%Y-%m-%d') + ' ' + df['laag_tijd'], errors='coerce')
df['hoog_tijd'] = df['hoog_tijd'].dt.tz_localize('UTC').dt.tz_convert(TIMEZONE)
df['laag_tijd'] = df['laag_tijd'].dt.tz_localize('UTC').dt.tz_convert(TIMEZONE)
df.drop(['datum'], axis=1, inplace=True)

df.to_csv('getijden.csv', index=False)
print('Stored cleaned tide table as getijden.csv')
df.head()


## 4. Build the common filters

We keep the useful season months and Belgian public holidays. July and August stay excluded by default because the coast is busier.

In [ ]:
unique_years = df['laag_tijd'].dropna().dt.year.unique().tolist()
belgian_holidays = Belgium(years=unique_years)
feestdagen = [str(day) for day in belgian_holidays.keys()]

month_filter = df['laag_tijd'].dt.month.isin(ALLOWED_MONTHS)
feestdag_filter = df['laag_tijd'].dt.date.astype(str).isin(feestdagen)

print(f"Belgian holidays loaded for: {unique_years}")
print(f"Allowed months: {ALLOWED_MONTHS}")


## 5. Activity-specific filter

In [ ]:
# Zanddam / dam schuppen: low tide between 15u00 and 17u00 local time.
low_tide_filter = (df['laag_tijd'].dt.hour >= 15) & (df['laag_tijd'].dt.hour < 17)

# Dam activity can also work with a long weekend rhythm: Friday, Saturday, Sunday or Monday,
# plus Belgian public holidays.
weekend_filter = df['laag_tijd'].dt.weekday.isin([4, 5, 6, 0])
weekend_feestdag = weekend_filter | feestdag_filter


## 6. Candidate days

In [ ]:
combined_filter = low_tide_filter & month_filter & weekend_feestdag

if FUTURE_ONLY:
    combined_filter = combined_filter & (df['laag_tijd'].dt.date >= date.today())

best_days = df[combined_filter].sort_values('laag_tijd').copy()
best_days


## 7. Human-readable result

In [ ]:
if best_days.empty:
    print("No candidate days found with these parameters.")
else:
    max_length = max([len(row['laag_tijd'].strftime("%A %d %B %Y")) for _, row in best_days.iterrows()])

    for _, row in best_days.iterrows():
        date_str = row['laag_tijd'].strftime("%A %d %B %Y")
        low_time = row['laag_tijd'].strftime("%Hu%M")
        high_time = row['hoog_tijd'].strftime("%Hu%M") if pd.notna(row['hoog_tijd']) else "onbekend"
        low_level = row['laag_water']
        print(f"{date_str:<{max_length}} (laag water {low_time}, hoog water {high_time}, laagwaterstand {low_level:.2f} m TAW)")
